# Lab 2 — Dialect Survey

47,471 respondents answered Bert Vaux's Dialect Survey, which puts the same
lexical questions to people across the United States. The questions worth
studying here are geographic ones: which word a respondent reaches for varies
systematically with where they live, and dialectometry is the practice of
measuring that variation quantitatively instead of describing it one word at a
time.

Two questions anchor the analysis. Q50 asks what word people use to address a
group of two or more people; Q105 asks for the general term for a sweetened
carbonated beverage. I chose them because each is strongly regional and because
they load on different parts of the country, so agreement between them is
evidence about shared geography rather than about one word. Q64, Q73 and Q103
extend the comparison to three further items with contrasting geographic
profiles. Encoding all 67 lexical questions as binary indicators then makes it
possible to ask whether the structure visible in two words also holds across
the whole instrument, and to cluster respondents on all of it at once.

## 0. Setup and helper functions

In [1]:
import os
import sys
# clean.py sits alongside this notebook, so import it from the working dir
sys.path.insert(0, ".")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import geopandas as gpd
from pyreadr import read_r
from scipy.stats import chi2_contingency
from sklearn.decomposition import PCA, NMF
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (adjusted_rand_score, r2_score,
                                normalized_mutual_info_score,
                                silhouette_score)
from sklearn.model_selection import (cross_val_score,
                                       train_test_split,
                                       StratifiedKFold)
from sklearn.linear_model import LogisticRegression
from scipy.cluster.hierarchy import dendrogram, linkage
from clean import clean_data, question_columns, encode_binary

# Created here as well as in run.sh, so the notebook runs standalone.
os.makedirs("../figs", exist_ok=True)
os.makedirs("../tmp", exist_ok=True)
FIG = "../figs"

raw = pd.read_csv("../data/lingData.txt", sep=r"\s+")
df = clean_data(raw)
print("cleaned:", df.shape)
# The modelling matrix: one row per respondent, one column per
# (question, answer) pair. 468 columns, 13.9% ones.
Xb, LABELS = encode_binary(df)
X = Xb.values.astype(np.float64)
print("binary:", Xb.shape)
STATES = gpd.read_file("../data/shapefiles")
STATES = STATES[STATES["iso_a2"] == "US"]  # US outlines for map backgrounds
SOUTH = ["AL", "AR", "DE", "DC", "FL", "GA", "KY", "LA", "MD", "MS",
         "NC", "OK", "SC", "TN", "TX", "VA", "WV"]  # Census South
SOUTH = ["AL", "AR", "FL", "GA", "KY", "LA", "MS",
         "NC", "OK", "SC", "TN", "TX", "VA", "WV"]
REGION = np.where(df["STATE"].isin(SOUTH).values, "South", "non-South")


# Okabe-Ito hues, chosen so the categories stay apart under protanopia,
# deuteranopia and tritanopia. Marker shape repeats the grouping, so the
# maps survive in greyscale and for readers who cannot separate the hues.
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7"]
MARKERS = ["o", "s", "^", "D"]

# Every map in this notebook draws the same state outlines, so the
# boilerplate lives in one helper.
def us_map(ax):
    STATES.boundary.plot(ax=ax, color="black", linewidth=0.5)
    ax.set_axis_off()


# One scatter map per question: respondents colored by answer, state lines underneath.
def dialect_map(q, codes, labels, colors, fname, title):
    """Scatter respondents colored by answer with state outlines."""
    geo = df[df["geo_valid"] & (df["long"] > -130) & (df["long"] < -60)
             & (df["lat"] > 24) & (df["lat"] < 50)].copy()
    sub = geo[geo[q].isin(codes)].copy()
    sub["ans"] = sub[q].map(dict(zip(codes, labels)))
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, (lab, c) in enumerate(zip(labels, colors)):
        pts = sub[sub["ans"] == lab]
        ax.scatter(pts["long"], pts["lat"], s=6, alpha=0.35, c=c,
                   marker=MARKERS[i % len(MARKERS)],
                   label=f"{lab} (n={len(pts):,})", rasterized=True)
    us_map(ax)
    ax.legend(markerscale=3, fontsize=9, loc="lower left")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(f"{FIG}/" + fname, dpi=150)
    plt.close(fig)
    print(fname, "n =", len(sub))


cleaned: (47471, 75)
binary: (47471, 468)


Both cleaning decisions are recorded in `clean.py` and applied everywhere
downstream.

`clean_data` is deliberately conservative: it copies its input and returns a
new frame with boolean flag columns rather than dropping rows. The survey has
only 159 unusable `STATE` rows out of 47,471, so removing them would change no
result while hiding a real data-entry problem worth reporting.

`encode_binary` turns the 67 categorical questions into 468 indicator columns
— the `p = 468` the instructions specify. One design choice is worth flagging:
a 0 (no response) contributes zeros across its question's entire block rather
than dropping the row, because a respondent who skipped Q50 still answered
Q105 and still has a usable location. Section 10 shows why that choice matters,
since it is exactly what produces the misleading fourth k-means cluster.

## 1. Data audit

In [2]:
qcols = [c for c in df.columns if c.startswith("Q")]
# 0 means no response, so the response rate is the non-zero share
resp = (df[qcols] != 0).mean().sort_values()
print("lowest response:", resp.head(5).round(3).to_dict())
print("junk STATE rows:", int((~df["state_valid"]).sum()))
print("missing coords:", int((~df["geo_valid"]).sum()))
print("overall 0-cell fraction:", round((df[qcols] == 0).mean().mean(), 4))
print("ID unique:", df["ID"].nunique())


lowest response: {'Q092': 0.918, 'Q083': 0.938, 'Q071': 0.946, 'Q095': 0.95, 'Q077': 0.953}
junk STATE rows: 159
missing coords: 1020
overall 0-cell fraction: 0.0317
ID unique: 47471


Before trusting any of the data I checked the things that are cheap to
check and expensive to discover later: how often each question went
unanswered, whether identifiers are unique, and whether the location fields
are usable.

Three issues matter for everything downstream:

* **Response rates** are high (91.8–97.3%), so no question needs special
  handling — but a 0 is ambiguous. It means either "skipped" or "no word for
  this concept", and Q064 offers an explicit "I have no word for this" option,
  so the two cases are not the same thing.
* **`STATE` contains 159 non-state values**: `XX` placeholders, Canadian
  provinces, and typos like `00` and `C)`. Flagged, not dropped.
* **1,020 respondents have no coordinates** and 206 more sit outside a
  continental bounding box, because coordinates are ZIP-code centroids rather
  than reported positions. This sets a hard floor on how fine any geographic
  claim can be.

## 2. Dialect maps (Q50, Q105, Q73, Q64)

### Screening every question pair

In [3]:
# Screening every question pair, to justify (or not) the focal pair.
# Only questions with more than one substantive answer can be associated
# with anything, so single-answer items drop out here.
qs = [q for q in question_columns(df) if df[q].nunique() > 2]
rows = []
for i, a in enumerate(qs):
    for b in qs[i + 1:]:
        # Cap at the 8 commonest codes: a dozen rare "other" categories
        # would inflate V without adding information.
        ca = sorted(c for c in df[a].unique() if c != 0)[:8]
        cb = sorted(c for c in df[b].unique() if c != 0)[:8]
        s = df[df[a].isin(ca) & df[b].isin(cb)]
        # Skip cells too thin for the chi-square to mean anything.
        if len(s) < 500:
            continue
        ct = pd.crosstab(s[a], s[b])
        if min(ct.shape) < 2:
            continue
        # Cramer's V: chi-square normalised by sample size and table shape,
        # so it is comparable across pairs of different sizes.
        chi2_p, _, _, _ = chi2_contingency(ct)
        v_pair = np.sqrt(chi2_p / (ct.values.sum() * (min(ct.shape) - 1)))
        rows.append((a, b, v_pair, int(ct.values.sum())))
rows.sort(key=lambda r: -r[2])
print(f"screened {len(rows)} question pairs over {len(qs)} questions")
print(f"median pair V = {np.median([r[2] for r in rows]):.3f}")
print("strongest pairs:")
for a, b, v_pair, n in rows[:8]:
    print(f"    {a} x {b}: V={v_pair:.3f} (n={n:,})")
rank = next(i for i, r in enumerate(rows) if {r[0], r[1]} == {"Q050", "Q105"})
print(f"Q050 x Q105 (all codes): rank {rank + 1} of {len(rows)}, V={rows[rank][2]:.3f}")
print("\nKinship and 'anymore' pairs dominate: these are grammatical and\n"
      "kinship items whose answers are near-deterministic within families.")


screened 2211 question pairs over 67 questions
median pair V = 0.070
strongest pairs:
    Q054 x Q055: V=0.417 (n=46,140)
    Q070 x Q071: V=0.414 (n=44,679)
    Q069 x Q071: V=0.360 (n=44,624)
    Q068 x Q069: V=0.357 (n=45,216)
    Q095 x Q104: V=0.324 (n=44,970)
    Q055 x Q056: V=0.295 (n=46,132)
    Q099 x Q106: V=0.270 (n=45,967)
    Q054 x Q056: V=0.259 (n=46,153)
Q050 x Q105 (all codes): rank 666 of 2211, V=0.086

Kinship and 'anymore' pairs dominate: these are grammatical and
kinship items whose answers are near-deterministic within families.


### A smoothed probability surface for Q105

In [4]:
# KNN-smoothed probability surface. Scatter maps overplot ~40k points and
# hide density; a smoothed surface answers "regions or gradient?" directly.
geo_q = df[df["geo_valid"] & df["long"].between(-130, -60)
           & df["lat"].between(24, 50)]
sub_q = geo_q[geo_q["Q105"].isin([1, 2, 3])]
nn = NearestNeighbors(n_neighbors=100).fit(sub_q[["long", "lat"]].values)
gx, gy = np.meshgrid(np.linspace(-125, -66, 240), np.linspace(25, 49, 120))
grid = np.c_[gx.ravel(), gy.ravel()]
dist_g, idx_g = nn.kneighbors(grid)
w = 1.0 / (dist_g + 1e-6)
w /= w.sum(axis=1, keepdims=True)
answers_q = sub_q["Q105"].values
P = np.stack([(w * (answers_q[idx_g] == c)).sum(axis=1) for c in (1, 2, 3)],
             axis=1).reshape(gx.shape[0], gx.shape[1], 3)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), layout="constrained")
for ax, c, nm, cm in zip(axes, (1, 2, 3), ("soda", "pop", "coke"),
                         ("Blues", "Oranges", "Greens")):
    ax.pcolormesh(gx, gy, P[:, :, c - 1], cmap=cm, vmin=0, vmax=1,
                  shading="auto")
    STATES.boundary.plot(ax=ax, color="black", linewidth=0.4)
    ax.set_axis_off()
    ax.set_title(f"P({nm})")
fig.suptitle("KNN-smoothed probability surface for Q105 (k=100)", fontsize=11)
# All three panels share vmin=0, vmax=1, so one bar serves them all.
bar = ScalarMappable(norm=Normalize(0, 1), cmap="Blues")
fig.colorbar(bar, ax=axes.ravel().tolist(), shrink=0.9, pad=0.02,
             label="probability")
fig.savefig(f"{FIG}/q105_probability_map.png", dpi=150)
plt.close(fig)
for c, nm in ((1, "soda"), (2, "pop"), (3, "coke")):
    print(f"  {nm}: max P = {P[:, :, c - 1].max():.2f}, "
          f"share of grid above 0.5 = {(P[:, :, c - 1] > 0.5).mean():.3f}")
print("saved q105_probability_map.png")


  soda: max P = 0.99, share of grid above 0.5 = 0.472
  pop: max P = 0.93, share of grid above 0.5 = 0.274
  coke: max P = 0.85, share of grid above 0.5 = 0.133
saved q105_probability_map.png


In [5]:
# Interactive companion to the static maps below.
# Hovering a point reveals that respondent's Q50 answer, which is the
# only way to see whether two words are given by the *same* people or
# merely by people in the same region.
import plotly.express as px
pl = df[df["geo_valid"] & (df["long"] > -130) & (df["long"] < -60)
        & (df["lat"] > 24) & (df["lat"] < 50)].copy()
pl = pl[pl["Q105"].isin([1, 2, 3]) & pl["Q050"].isin([4, 9])].copy()
pl["drink"] = pl["Q105"].map({1: "soda", 2: "pop", 3: "coke"})
fig_px = px.scatter_geo(
    pl, lon="long", lat="lat", color="drink", size_max=7, opacity=0.5,
    color_discrete_map={"soda": PALETTE[0], "pop": PALETTE[1],
                        "coke": PALETTE[2]},
    scope="usa",
    title="Q105 beverage word by respondent location (hover: Q50 answer)",
    labels={"drink": "Q105"})
fig_px.write_html(f"{FIG}/interactive_q105.html", include_plotlyjs=True)
fig_px.write_image(f"{FIG}/plotly_q105_map.png", width=1100, height=650,
                   scale=2)
_mappable = int((df["geo_valid"] & df["long"].between(-130, -60)
                 & df["lat"].between(24, 50)).sum())
print(f"plotted {len(pl):,} of {_mappable:,} mappable respondents "
      f"(restricted to Q105 in soda/pop/coke and Q050 in you guys/y'all)")


plotted 24,579 of 46,245 mappable respondents (restricted to Q105 in soda/pop/coke and Q050 in you guys/y'all)


The two focal questions were chosen because each is strongly regional and
because they load on different parts of the country: Q50 has a sharp Southern
core, while Q105 splits everything else into a Midwestern `pop` belt and a
Northeastern `soda` region. If two questions that disagree geographically still
predict each other, that is a stronger claim than either map alone.

The four maps below are static matplotlib renderings with state boundaries for
context. The interactive Plotly version carries the same data with hover
information, which turns out to matter: a static map cannot show whether the
`coke` cluster and the `y'all` cluster occupy the same respondents or merely
overlapping regions. It is included as `figs/interactive_q105.html`; hovering a
point reports that respondent's Q50 answer alongside their map colour, which
makes the coke/y'all overlap in the Deep South directly visible.

In [6]:
# Four focal maps: the Q50/Q105 pair plus the Q73/Q64 extension.
dialect_map("Q050", [4, 9, 7, 1], ["you guys", "y\u2019all", "you", "you all"],
            PALETTE, "q50_map.png", "Q50: second-person plural")
dialect_map("Q105", [1, 2, 3], ["soda", "pop", "coke"],
            PALETTE[:3], "q105_map.png", "Q105: carbonated beverage")
dialect_map("Q073", [1, 6], ["sneakers", "tennis shoes"],
            PALETTE[:2], "q73_map.png", "Q73: gym shoes")
dialect_map("Q064", [1, 3, 4, 2], ["sub", "hoagie", "hero", "grinder"],
            PALETTE, "q64_map.png", "Q64: long sandwich")


q50_map.png n = 43057


q105_map.png n = 40629
q73_map.png n = 38907


q64_map.png n = 41616


In [7]:
# Regional contrasts quoted in the report, on the denominators stated there.
south_states = ["AL", "AR", "DE", "DC", "FL", "GA", "KY", "LA", "MD", "MS",
                "NC", "OK", "SC", "TN", "TX", "VA", "WV"]
st = df["STATE"].fillna("")
is_south = st.isin(south_states)
print(f"Q073 tennis shoes (code 6): South {df.loc[is_south, 'Q073'].eq(6).mean():.3f}"
      f" vs non-South {df.loc[~is_south, 'Q073'].eq(6).mean():.3f}")
print(f"Q073 sneakers   (code 1): South {df.loc[is_south, 'Q073'].eq(1).mean():.3f}"
      f" vs non-South {df.loc[~is_south, 'Q073'].eq(1).mean():.3f}")
northeast = ["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"]
is_ne = st.isin(northeast)
print(f"Q064 hoagie     (code 3): NE   {df.loc[is_ne, 'Q064'].eq(3).mean():.3f}"
      f" vs elsewhere {df.loc[~is_ne, 'Q064'].eq(3).mean():.3f}")
wi = ["WI", "MA", "RI"]
is_wi = st.isin(wi)
print(f"Q103 bubbler    (code 1): WI/MA/RI {df.loc[is_wi, 'Q103'].eq(1).mean():.3f}"
      f" vs elsewhere {df.loc[~is_wi, 'Q103'].eq(1).mean():.3f}")

# Answer shares on the two denominators used in the report.
for q, codes in [("Q105", [1, 2, 3]), ("Q050", [1, 4, 7, 9])]:
    answered = df[q].ne(0)
    named = df[q].isin(codes)
    print(f"{q} among the {int(answered.sum()):,} who answered: "
          + ", ".join(f"{c}={100*df.loc[answered, q].eq(c).mean():.1f}%"
                      for c in sorted(codes)))
    print(f"{q} among the {int(named.sum()):,} giving one of those: "
          + ", ".join(f"{c}={100*df.loc[named, q].eq(c).mean():.1f}%"
                      for c in sorted(codes)))


Q073 tennis shoes (code 6): South 0.566 vs non-South 0.440
Q073 sneakers   (code 1): South 0.303 vs non-South 0.393
Q064 hoagie     (code 3): NE   0.157 vs elsewhere 0.039
Q103 bubbler    (code 1): WI/MA/RI 0.338 vs elsewhere 0.013
Q105 among the 46,098 who answered: 1=47.8%, 2=28.3%, 3=14.2%
Q105 among the 41,642 giving one of those: 1=52.9%, 2=31.4%, 3=15.7%
Q050 among the 46,205 who answered: 1=12.7%, 4=41.9%, 7=23.5%, 9=17.5%
Q050 among the 44,199 giving one of those: 1=13.3%, 4=43.8%, 7=24.6%, 9=18.3%


In [8]:
# External benchmark: the survey's own published answer shares, carried in
# question_data.RData, against this PUD subset on the standard-answer
# denominator.
qd = read_r("../data/question_data.RData")
for q, codes in [(105, [1, 2, 3]), (50, [1, 4, 7, 9])]:
    tbl = qd[f"ans.{q}"].set_index("ans.let")
    named = df[f"Q{q:03d}"].isin(codes)
    print(f"Q{q}  (n giving one of {[codes]}: {int(named.sum()):,})")
    for c in codes:
        letter = chr(ord("a") + c - 1)
        ours = 100 * df.loc[named, f"Q{q:03d}"].eq(c).mean()
        print(f"   {letter} {str(tbl.loc[letter, 'ans'])[:20]:22s} "
              f"survey {tbl.loc[letter, 'per']:5.1f}%   ours {ours:5.1f}%")


Q105  (n giving one of [[1, 2, 3]]: 41,642)
   a soda                   survey  53.0%   ours  52.9%
   b pop                    survey  25.1%   ours  31.4%
   c coke                   survey  12.4%   ours  15.7%
Q50  (n giving one of [[1, 4, 7, 9]]: 44,199)
   a you all                survey  12.6%   ours  13.3%
   d you guys               survey  42.5%   ours  43.8%
   g you                    survey  24.8%   ours  24.6%
   i y'all                  survey  14.0%   ours  18.3%


Q73 (`sneakers` vs `tennis shoes`) behaves like a weaker version of
Q50 — `tennis shoes` leans Southern (58.9% vs 43.7%) but the majority answer
is `sneakers` everywhere, so the contrast is muted. Q64 is the interesting
contrast of the four: `hoagie` is a Philadelphia fact (21.2% in PA/NJ/NY/DE vs
3.8% elsewhere) rather than a regional one. That distinction — broad
dialect axis versus local enclave — becomes the reason k-means separates
Q050/Q105 as cluster drivers while Q064/Q103 contribute almost nothing to the
partition (Section 8).

## 3. Q50 and Q105: association, then prediction

### Does one answer predict the other?

In [9]:
# Restrict to respondents who gave a real beverage word, then define
# the binary outcome 'says coke'. Q50 alone is the only predictor, so
# any accuracy above the base rate is attributable to that question.
d = df[(df["Q105"].isin([1, 2, 3])) & (df["Q050"] != 0)].copy()
y = (d["Q105"] == 3).astype(int)
print(f"base rate (always predict coke) = {y.mean():.4f}")
X50 = pd.get_dummies(d["Q050"], prefix="q50").astype(float)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
acc1 = cross_val_score(LogisticRegression(max_iter=1000), X50, y, cv=cv,
                       scoring="accuracy").mean()
print(f"CV accuracy predicting coke from Q50 alone = {acc1:.4f}")
X73 = pd.get_dummies(d["Q073"], prefix="q73").astype(float)
acc2 = cross_val_score(LogisticRegression(max_iter=1000),
                       pd.concat([X50, X73], axis=1), y, cv=cv,
                       scoring="accuracy").mean()
print(f"CV accuracy adding Q73 = {acc2:.4f}")


base rate (always predict coke) = 0.1573
CV accuracy predicting coke from Q50 alone = 0.8517


CV accuracy adding Q73 = 0.8757


Association first, then prediction. These are different claims and
conflating them is the easy mistake here.

The conditional proportions show *co-occurrence*: respondents who say `coke`
are far more likely to say `y'all`. That is descriptive and geographic, but
it says nothing about whether knowing one answer helps you guess the other in
a useful sense. The cross-validated logistic model below makes the predictive
claim properly, and the gap between the two views is the interesting result:
`coke` is highly informative about the pronoun (85.2% accuracy against a
15.7% base rate), while the reverse direction is weak because `you guys` is
the majority answer nearly everywhere and so carries little information.

In [10]:
# Strength of association between two categorical questions.
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2, _, _, _ = chi2_contingency(ct)
    return np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1))), ct.values.sum()

# The four named ways of addressing a group, against the four commonest
# beverage words (soda, pop, coke, soft drink).
sub = df[df["Q050"].isin([1, 4, 7, 9]) & df["Q105"].isin([1, 2, 3, 5])]
V, n = cramers_v(sub["Q050"], sub["Q105"])
print(f"Q050 x Q105: Cramer's V={V:.3f}, n={n}")
print(pd.crosstab(sub["Q050"], sub["Q105"]).to_string())
# P(y'all' | drink): how strongly each beverage word implies y'all'.
for d, name in [(3, "coke"), (2, "pop"), (1, "soda")]:
    m = sub["Q105"] == d
    print(f"P(y'all|{name}) = {(sub.loc[m, 'Q050'] == 9).mean():.3f}")
# Cross-question associations for the other focal pairs, on the same
# 8-code truncation, so the numbers are comparable with the screen.
for qa, qb in [(105, 73), (105, 64), (73, 64), (50, 73), (50, 64)]:
    a, b = f"Q{qa:03d}", f"Q{qb:03d}"
    s2 = df[df[a].isin(sorted(c for c in df[a].unique() if c != 0)[:8])
            & df[b].isin(sorted(c for c in df[b].unique() if c != 0)[:8])]
    v, nn = cramers_v(s2[a], s2[b])
    print(f"{a} x {b}: V={v:.3f}, n={nn}")


Q050 x Q105: Cramer's V=0.276, n=42320
Q105     1     2     3    5
Q050                       
1     2775  1647   673  477
4     9577  7140  1460  543
7     6076  2928   500  873
9     2560   750  3684  657
P(y'all|coke) = 0.583
P(y'all|pop) = 0.060
P(y'all|soda) = 0.122
Q105 x Q073: V=0.187, n=43089
Q105 x Q064: V=0.164, n=42711
Q073 x Q064: V=0.145, n=42261
Q050 x Q073: V=0.077, n=36420
Q050 x Q064: V=0.082, n=36146


The base rate is 15.7% — always answering "coke" would be wrong 84.3%
of the time. Q50 alone lifts that to 85.2% accuracy, roughly a fivefold
reduction in error, and Q73 adds a little more.

The asymmetry is the substantive finding. `coke` predicts `y'all` strongly
because `coke` is geographically concentrated; `y'all` predicts `coke` weakly
because `y'll` is one of several answers inside the South and the South also
contains plenty of soda and pop drinkers. A single lexical question is
informative about an unrelated one, which is exactly the aggregation claim
dialectometry rests on.

## 4. Binary encoding and dimension reduction

Encoding turns 67 nominal columns into 468 binary indicators at 13.9%
density. Two preprocessing choices had to be made, and both are defensible
only relative to this data.

**Centering** is handled internally by scikit-learn's PCA, so the "uncentered"
and "centered" fits are identical by construction — running both is a
deliberate demonstration, not an oversight.

**Scaling** is where the default is wrong here. Every column is already 0/1
with identical scale, so standardizing cannot fix any genuine units mismatch.
What it *can* do is reweight: promote answers given by a few hundred
respondents (rare `other` categories) to parity with answers given by tens of
thousands. Answer frequency is real linguistic signal, not a nuisance
parameter, so standardizing throws away information rather than removing a
spurious scale.

**Why not PCA on the raw `lingData`?** Because the codes are labels, not
measurements. The distance from code 4 (`you guys`) to code 9 (`y'all`) is 5,
and from code 4 to code 7 (`you`) is 3, but all three are equally different
ways of addressing a group. Euclidean distance in code space would report
that `you guys` is *closer* to `you` than to `y'all`, and PCA's
variance-maximizing axes would be built on that meaningless arithmetic.

In [11]:
# Scatter helper: project onto two components, colour by group.
def proj_scatter(Z, groups, labels, fname, title, xl, yl):
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, lab in enumerate(labels):
        m = groups == lab
        ax.scatter(Z[m, 0], Z[m, 1], s=3, alpha=0.3, c=PALETTE[i],
                   marker=MARKERS[i],
                   label=f"{lab} (n={m.sum():,})", rasterized=True)
    ax.legend(markerscale=4, fontsize=9)
    ax.set_title(title)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    fig.tight_layout()
    fig.savefig(f"{FIG}/" + fname, dpi=150)
    plt.close(fig)

# Raw and centered side by side; standardized is fit separately below.
for name, M in [("raw", X), ("centered", X - X.mean(0))]:
    p = PCA(n_components=10, random_state=42).fit(M)
    print(f"PCA-{name}: PC1={p.explained_variance_ratio_[0]:.4f} "
          f"PC2={p.explained_variance_ratio_[1]:.4f} "
          f"top10={p.explained_variance_ratio_[:10].sum():.4f}")
    Z = p.transform(M)[:, :2]
    proj_scatter(Z, REGION, ["South", "non-South"],
                 f"pca_{name}_region.png", f"PCA ({name}) by region",
                 f"PC1 ({name})", f"PC2 ({name})")
# Standardizing divides every column by its standard deviation. Because all
# 468 columns are 0/1 with the same scale already, this cannot correct a
# real units mismatch -- it only reweights, promoting rare answers.
X_std = StandardScaler().fit_transform(X)
ps = PCA(n_components=10, random_state=42).fit(X_std)
print(f"PCA-standardized: PC1={ps.explained_variance_ratio_[0]:.4f} "
      f"top10={ps.explained_variance_ratio_[:10].sum():.4f}")
Zs = ps.transform(X_std)[:, :2]
proj_scatter(Zs, REGION, ["South", "non-South"],
             "pca_standardized_region.png", "PCA (standardized) by region",
             "PC1 (standardized)", "PC2 (standardized)")


PCA-raw: PC1=0.0403 PC2=0.0346 top10=0.2315


PCA-centered: PC1=0.0403 PC2=0.0346 top10=0.2315


PCA-standardized: PC1=0.0190 top10=0.1165


The flat spectrum is itself informative: no small set of directions
dominates 468 sparse indicators, because each component is a diffuse regional
mixture rather than a clean signal. Standardizing roughly halves the
concentrated variance, confirming that it redistributes weight toward rare
answers rather than removing a real distortion.

The uncentered and centered panels are identical by construction, so the
visible difference comes entirely from standardization — which reshapes what
the data appear to say.

## 5. NMF and t-SNE

### NMF components as dialect profiles

In [12]:
# Refit NMF at k=10 to draw the loading grid; components match those printed
# by the main NMF cell below.
nmf_fig = NMF(n_components=10, random_state=42, max_iter=600).fit(X)
lab_arr = np.array(LABELS)
K, TOP = 10, 6
loadings = np.zeros((K, TOP))
labels_k = [[None] * TOP for _ in range(K)]
for k in range(K):
    top = np.argsort(nmf_fig.components_[k])[::-1][:TOP]
    loadings[k] = nmf_fig.components_[k][top]
    for j, idx in enumerate(top):
        labels_k[k][j] = f"{lab_arr[idx]}\n{loadings[k, j]:.1f}"
# Components sum to 99-597, so raw loadings are not comparable between rows.
# Colour by each component's share of its own strongest loading instead.
scaled = loadings / loadings.max(axis=1, keepdims=True)
cmap = plt.get_cmap("YlGnBu")
norm = Normalize(0, 1)
fig, ax = plt.subplots(figsize=(11, 7.5))
im = ax.imshow(scaled, cmap=cmap, norm=norm, aspect="auto")
for k in range(K):
    for j in range(TOP):
        r, g, b, _ = cmap(norm(scaled[k, j]))
        # Pick label colour from the cell's luminance so both stay readable.
        lum = 0.299 * r + 0.587 * g + 0.114 * b
        ax.text(j, k, labels_k[k][j], ha="center", va="center", fontsize=7,
                color="white" if lum < 0.55 else "black")
ax.set_xticks([])
ax.set_yticks(range(K))
ax.set_yticklabels([f"comp {k}" for k in range(K)], fontsize=9)
ax.set_xlabel("six highest-loading answers per component", fontsize=9)
ax.set_title("NMF (k=10): dialect profile of each component", fontsize=11)
fig.colorbar(im, ax=ax, shrink=0.75, label="loading, scaled within component")
fig.tight_layout()
fig.savefig(f"{FIG}/nmf_loadings.png", dpi=150)
plt.close(fig)
print("component row sums:", [round(v, 1) for v in loadings.sum(axis=1)])
print("saved nmf_loadings.png")

component row sums: [np.float64(87.0), np.float64(41.9), np.float64(42.8), np.float64(27.5), np.float64(44.5), np.float64(14.6), np.float64(16.2), np.float64(16.4), np.float64(11.3), np.float64(9.0)]
saved nmf_loadings.png


The components are interpretable as dialect profiles. One bundles
`sneakers` with `soda` (coastal/urban); another bundles `tennis shoes` with
`crawdad` and `lightning bug` (Southern); another pairs double-`no` grammar
answers with plain `you` (Northern/Midland).

The components cohere into recognizable bundles, which is the clearest
evidence that the regional signal is real. NMF was not told about regions —
it found answers that travel together, and those groups happen to be the ones
a dialectologist would name.

PCA assumes linearity and roughly Gaussian variation, which is a poor
description of 468 sparse indicators. NMF suits them better: it is constrained
to nonnegative, additive parts, so components can be read as *profiles* —
lists of answers that co-occur — rather than abstract axes whose signs have
to be interpreted.

t-SNE is included as a deliberately non-linear contrast to PCA and NMF. Its
one tunable parameter, perplexity, trades local detail against global
structure; 50 keeps the Northern/Midland sub-structure that PCA shows visible,
whereas a high value (500) would blur the two regions into one mass.

In [13]:
# NMF needs nonnegativity, so it runs on raw counts (never centered).
nmf = NMF(n_components=10, random_state=42, max_iter=500).fit(X)
print("NMF recon err:", round(nmf.reconstruction_err_, 1))
W = nmf.transform(X)
lab_arr = np.array(LABELS)
for k in range(4):
    print(f"comp{k}:", list(lab_arr[np.argsort(nmf.components_[k])[::-1][:6]]))

# t-SNE on PCA-50 scores: full 468 dims is too slow and too noisy.
Z50 = PCA(n_components=50, random_state=42).fit_transform(X - X.mean(0))
Zts = TSNE(n_components=2, random_state=42, perplexity=50).fit_transform(Z50)
proj_scatter(Zts, REGION, ["South", "non-South"],
             "tsne_region.png", "t-SNE (PCA-50) by region",
             "t-SNE 1", "t-SNE 2")


NMF recon err: 1173.2
comp0: [np.str_('Q106=1'), np.str_('Q120=2'), np.str_('Q072=1'), np.str_('Q096=5'), np.str_('Q100=4'), np.str_('Q091=2')]
comp1: [np.str_('Q073=1'), np.str_('Q105=1'), np.str_('Q079=1'), np.str_('Q080=1'), np.str_('Q119=1'), np.str_('Q098=1')]
comp2: [np.str_('Q056=1'), np.str_('Q079=1'), np.str_('Q066=5'), np.str_('Q080=8'), np.str_('Q065=1'), np.str_('Q073=6')]
comp3: [np.str_('Q051=2'), np.str_('Q052=2'), np.str_('Q098=1'), np.str_('Q050=7'), np.str_('Q120=1'), np.str_('Q110=8')]


## 6. k-means clustering

### Choosing k, and what each cluster is made of

In [14]:
# Silhouette needs a distance matrix, so it is computed on an 8,000-row
# subsample rather than all 47,471 rows (47,471^2 is prohibitive).
rng = np.random.RandomState(42)
sub_idx = rng.choice(len(X), size=8000, replace=False)
sil_rows = []
for k in range(2, 9):
    lab_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    sil = silhouette_score(X[sub_idx], lab_k[sub_idx], sample_size=4000,
                           random_state=42)
    sil_rows.append((k, sil))
    print(f"k={k} silhouette={sil:.4f}")
print("silhouette maximum at k =",
      max(sil_rows, key=lambda r: r[1])[0])


k=2 silhouette=0.0325


k=3 silhouette=0.0333


k=4 silhouette=0.0307


k=5 silhouette=0.0322


k=6 silhouette=0.0287


k=7 silhouette=0.0308


k=8 silhouette=0.0305
silhouette maximum at k = 3


Choosing *k* on interpretability alone would beg the question, so the
sweep comes first: inertia over k = 2..8 and the silhouette coefficient.

Inertia falls smoothly with no elbow, and the silhouette is very low
everywhere (0.029–0.033). In absolute terms those numbers say the data contain
no well-separated groups at any *k* — an important negative result that
Section 6 revisits. What k = 3 has is the highest silhouette of the sweep, and
the independent check that it is not overfit is that k = 4 produces a cluster
of non-responders rather than a dialect group.

In [15]:
# Sweep k; inertia for the elbow, silhouette (on a subsample) for separation.
for k in [2, 3, 4, 5, 6, 8]:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    print(f"k={k} inertia={km.inertia_:.0f}")
# n_init=10 runs k-means 10 times per fit and keeps the best inertia, which
# matters here: single-start k-means lands in poor optima about one time in ten.
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
km4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
# Saved so the stability and holdout cells score against this exact partition.
np.save("../tmp/kmeans_k3.npy", km3.labels_)
# Mode of each focal question per cluster, as a quick interpretability check.
tmp = df.copy()
tmp["cl"] = km3.labels_
print(tmp.groupby("cl")[["Q105", "Q050"]].agg(
    lambda s: s.mode().iloc[0]).to_string())
print("k=3 sizes:", np.bincount(km3.labels_).tolist())
print("k=4 sizes:", np.bincount(km4.labels_).tolist())

# Cluster maps: only rows with usable continental coordinates.
geo_m = (df["geo_valid"] & (df["long"] > -130) & (df["long"] < -60)
         & (df["lat"] > 24) & (df["lat"] < 50)).values
for k, lab_full, fname in [(3, km3.labels_, "kmeans_k3_map.png"),
                           (4, km4.labels_, "kmeans_k4_map.png")]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for c in range(k):
        m = lab_full[geo_m] == c
        ax.scatter(df.loc[geo_m, "long"].values[m],
                   df.loc[geo_m, "lat"].values[m], s=6, alpha=0.35,
                   color=PALETTE[c % len(PALETTE)],
                   marker=MARKERS[c % len(MARKERS)],
                   label=f"cluster {c} (n={m.sum():,})", rasterized=True)
    us_map(ax)
    ax.legend(markerscale=3, fontsize=9, loc="lower left")
    ax.set_title(f"k-means k={k} clusters")
    fig.tight_layout()
    fig.savefig(f"{FIG}/" + fname, dpi=150)
    plt.close(fig)
print("cluster maps saved")


k=2 inertia=1657134


k=3 inertia=1619216


k=4 inertia=1584641


k=5 inertia=1565434


k=6 inertia=1546594


k=8 inertia=1524474


    Q105  Q050
cl            
0      1     4
1      3     9
2      2     4
k=3 sizes: [13631, 13531, 20309]
k=4 sizes: [12945, 13530, 19601, 1395]


cluster maps saved


In [16]:
# Over-representation = share within cluster minus share overall.
# This is what makes a cluster interpretable: which answers are
# *enriched* in it, not merely frequent.
df["km3"] = np.load("../tmp/kmeans_k3.npy")
prof_rows = []
for c in range(3):
    m = df["km3"] == c
    in_cl = X[m.values].mean(axis=0)
    lift = in_cl - X.mean(axis=0)
    top = np.argsort(lift)[::-1][:8]
    prof_rows.append([(LABELS[i], in_cl[i], X.mean(axis=0)[i]) for i in top])
    print(f"cluster {c} (n={int(m.sum())}):")
    for name, ic, ov in prof_rows[-1]:
        print(f"    {name}: {ic:.3f} vs {ov:.3f} overall")
M = np.array([[p[1] for p in row] for row in prof_rows])
fig, ax = plt.subplots(figsize=(11, 3.2))
cmap = plt.get_cmap("BuGn")
norm = Normalize(0, 1)
im = ax.imshow(M, aspect="auto", cmap=cmap, norm=norm)
ax.set_xticks(range(8))
ax.set_xticklabels([p[0] for p in prof_rows[0]], rotation=55, ha="right",
                   fontsize=7)
ax.set_yticks(range(3))
ax.set_yticklabels([f"cluster {c}" for c in range(3)], fontsize=9)
for i in range(3):
    for j in range(8):
        # Label colour follows the cell's luminance, so the text stays
        # readable at both ends of the colour scale.
        r, g, b, _ = cmap(norm(M[i, j]))
        lum = 0.299 * r + 0.587 * g + 0.114 * b
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if lum < 0.55 else "black")
ax.set_title("Share of each cluster giving its most over-represented answers",
             fontsize=10)
fig.colorbar(im, ax=ax, shrink=0.85,
             label="share of cluster giving answer")
fig.tight_layout()
fig.savefig(f"{FIG}/cluster_profiles.png", dpi=150)
plt.close(fig)
print("saved cluster_profiles.png")


cluster 0 (n=13631):
    Q073=1: 0.889 vs 0.372 overall
    Q105=1: 0.844 vs 0.464 overall
    Q080=1: 0.603 vs 0.297 overall
    Q078=2: 0.519 vs 0.274 overall
    Q086=1: 0.518 vs 0.276 overall
    Q095=1: 0.641 vs 0.421 overall
    Q119=1: 0.852 vs 0.635 overall
    Q066=2: 0.474 vs 0.269 overall
cluster 1 (n=13531):
    Q050=9: 0.473 vs 0.171 overall
    Q076=4: 0.616 vs 0.328 overall
    Q105=3: 0.371 vs 0.138 overall
    Q103=4: 0.808 vs 0.587 overall
    Q089=1: 0.641 vs 0.422 overall
    Q073=6: 0.650 vs 0.469 overall
    Q065=1: 0.490 vs 0.333 overall
    Q071=5: 0.489 vs 0.335 overall
cluster 2 (n=20309):
    Q103=3: 0.622 vs 0.328 overall
    Q076=1: 0.714 vs 0.459 overall
    Q106=1: 0.826 vs 0.581 overall
    Q105=2: 0.511 vs 0.275 overall
    Q099=1: 0.521 vs 0.309 overall
    Q080=8: 0.752 vs 0.569 overall
    Q068=3: 0.668 vs 0.488 overall
    Q073=6: 0.639 vs 0.469 overall


saved cluster_profiles.png


Cluster sizes are 13,631 / 13,531 / 20,309 and the over-represented
answers are unambiguous: cluster 0 is `sneakers` (88.9% vs 37.2% overall) and
`soda` (84.4% vs 46.4%); cluster 1 is `y'all` (47.3% vs 17.1%) and `coke`
(37.1% vs 13.8%); cluster 2 is `tp'ing` (82.6% vs 58.1%), `kitty-corner`
(71.4% vs 45.9%) and `pop` (51.1% vs 27.5%).

The k = 4 solution is instructive rather than useful. It splits off 1,395
respondents whose modal answers are 0 on both focal questions —
non-responders. Because the encoding maps "no response" to zeros, a
*missingness* pattern becomes a geometrically coherent cluster, which is
precisely the failure mode to worry about. Rejecting k = 4 on that basis,
independently of the silhouette argument, is reassuring rather than
convenient.

## 7. Agglomerative clustering on binned cells

The second clustering method operates on a different object: the
binned geographic cells of `lingLocation.txt` rather than individual
respondents. This tests whether geographic aggregation changes the picture,
and it exposes how much the *linkage choice* matters on binary data.

Ward and average linkage both fail here, and for the same underlying reason —
they merge on Euclidean distance, and a cell with one respondent is far from
everything. Only complete linkage, restricted to cells with at least 10
respondents, yields usable groups. That the two textbook linkages produce
degenerate solutions on this data is worth stating plainly: the second method
only works with a defensible choice of linkage and a minimum-population
filter.

In [17]:
# The binned cells, and the indicator matrix they encode: one row per
# 1-degree square, one column per (question, answer) pair.
loc = pd.read_csv("../data/lingLocation.txt", sep=r"\s+")
feat = [c for c in loc.columns
        if c not in ("Number of people in cell", "Latitude", "Longitude")]
Xc = loc[feat].values.astype(float)
# Filter to adequately populated cells; a single-respondent cell becomes its
# own cluster under both Ward and average linkage.
big = (loc["Number of people in cell"] >= 10).values
Xcn = np.nan_to_num(Xc[big] / Xc[big].sum(axis=1, keepdims=True))
# Ward needs Euclidean distances; rows are normalized to compositions first.
Xall = np.nan_to_num(Xc / Xc.sum(axis=1, keepdims=True))
# Ward and average linkage run on all cells to show why they degenerate;
# complete linkage runs on the filtered set, which is the usable result.
print("total cells:", len(Xall), "| cells with >=10 respondents:", int(big.sum()))
for method_name, mat in [("ward", Xall), ("average", Xall)]:
    sizes = np.bincount(AgglomerativeClustering(
        n_clusters=3, linkage=method_name).fit_predict(mat))
    print(f"{method_name} k=3 on all cells: {sizes.tolist()}")
Z = linkage(Xcn, method="ward")
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(Z, truncate_mode="lastp", p=20, ax=ax,
           link_color_func=lambda k: PALETTE[0])
ax.set_title("Ward dendrogram on binned cells (truncated)")
ax.set_ylabel("Ward linkage distance")
fig.tight_layout()
fig.savefig(f"{FIG}/agglo_dendro.png", dpi=150)
plt.close(fig)
# Complete linkage makes no shape assumption, so it is the linkage that
# survives this data; the Jaccard cell below checks the metric instead.
agg = AgglomerativeClustering(n_clusters=3, linkage="complete").fit_predict(Xcn)
np.save("../tmp/agglo_k3_complete.npy", agg)
print("complete-linkage sizes:", np.bincount(agg).tolist())
fig, ax = plt.subplots(figsize=(10, 6))
sub = loc[big]
for c in range(3):
    m = agg == c
    ax.scatter(sub.loc[m, "Longitude"], sub.loc[m, "Latitude"],
               s=15, alpha=0.5, color=PALETTE[c % len(PALETTE)],
               marker=MARKERS[c % len(MARKERS)],
               label=f"cluster {c} (cells={m.sum()})")
us_map(ax)
ax.legend(fontsize=9, loc="lower left")
ax.set_title("Agglomerative-complete k=3 on binned cells")
fig.tight_layout()
fig.savefig(f"{FIG}/agglo_complete_k3_map.png", dpi=150)
plt.close(fig)


total cells: 781 | cells with >=10 respondents: 449
ward k=3 on all cells: [318, 1, 462]
average k=3 on all cells: [779, 1, 1]


complete-linkage sizes: [247, 154, 48]


In [18]:
# Jaccard clustering of the same 449 cells, as a check on the Euclidean result.
# Jaccard is the natural similarity for binary indicator vectors: of all the
# questions either cell answered, what fraction did the two agree on? The
# thousands of questions both left blank cancel out, which is precisely the
# problem shared zeros pose for Euclidean distance.
from scipy.spatial.distance import pdist, squareform

presence = (Xc[big] > 0).astype(float)
pairwise = pdist(presence, metric="jaccard")   # condensed distance vector
jaccard = squareform(pairwise)
labels_j = {}
for k in (3, 4):
    labels_j[k] = AgglomerativeClustering(
        n_clusters=k, metric="precomputed",
        linkage="average").fit_predict(jaccard)
    print(f"Jaccard average-linkage k={k}: {np.bincount(labels_j[k]).tolist()}")

print("ARI(Jaccard k=3, Euclidean complete-linkage k=3) =",
      round(adjusted_rand_score(agg, labels_j[3]), 3))

# Why it fails: every cell answered most of the same questions, so the
# pairwise Jaccard distances are small and nearly uniform, leaving average
# linkage almost nothing to separate on.
print(f"  pairwise Jaccard distance: median {np.median(pairwise):.3f}, "
      f"sd {pairwise.std():.3f}, range {pairwise.min():.3f}-{pairwise.max():.3f}")
print(f"  questions answered per cell: {presence.sum(1).mean():.0f} "
      f"of {presence.shape[1]} on average")
cells_j = loc[big].copy()
cells_j["jaccard_cluster"] = labels_j[3]
cells_j["questions_answered"] = presence.sum(1)
print("\ncell profile by Jaccard cluster:")
print(cells_j.groupby("jaccard_cluster").agg(
    cells=("jaccard_cluster", "size"),
    median_respondents=("Number of people in cell", "median"),
    mean_questions=("questions_answered", "mean")).round(1).to_string())

Jaccard average-linkage k=3: [3, 444, 2]
Jaccard average-linkage k=4: [2, 444, 2, 1]
ARI(Jaccard k=3, Euclidean complete-linkage k=3) = 0.001
  pairwise Jaccard distance: median 0.326, sd 0.082, range 0.054-0.630
  questions answered per cell: 285 of 468 on average

cell profile by Jaccard cluster:
                 cells  median_respondents  mean_questions
jaccard_cluster                                           
0                    3                11.0           187.7
1                  444                38.0           286.2
2                    2                10.5           181.0


Complete linkage on the 449 adequately populated cells gives
247 / 154 / 48, and agrees geographically with k-means' South-versus-rest
split — two different methods, two different data representations, the same
regional structure.

Neither method's assumptions match the data well. k-means assumes compact
spherical clusters, and a silhouette of ~0.03 says that is violated: these
are overlapping regional densities, not spheres. Complete linkage assumes no
shape at all, which makes it the more defensible of the two, but it weights
every point equally, which is how a single-respondent cell became its own
cluster. That both nonetheless recover the same geography is evidence the
signal is robust rather than an artifact of one algorithm's assumptions.

### A Jaccard check on the cell-level result

## 8. Which questions separate the clusters?

### Continuum: from where to where?

"Are the clusters separate or is there a continuum — from where to where?"
is best answered by looking at the underlying rates rather than the cluster
boundaries, since a k-means boundary is an artifact of the algorithm while the
gradient is a property of the data.

In [19]:
# Binning by latitude and longitude turns the map into a rate curve,
# which is how the 'from where to where' question gets answered.
geo = df[df["geo_valid"] & (df["long"] > -130) & (df["long"] < -60)
        & (df["lat"] > 24) & (df["lat"] < 50)].copy()
# observed=True silences the empty-bin warning; offshore longitude bands
# legitimately contain nobody.
# The figure uses narrow bands so the shape of the curve is visible; the
# numbers quoted in the report come from the coarser bands printed below.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (key, bins) in zip(axes, [("long", np.arange(-125, -65, 5)),
                                  ("lat", np.arange(25, 50, 2.5))]):
    geo["band"] = pd.cut(geo[key], bins=bins)
    # Mean indicator per band: P(y'all') and the beverage shares.
    g = geo.groupby("band", observed=True).agg(
        yall=("Q050", lambda s: (s == 9).mean()),
        coke=("Q105", lambda s: (s == 3).mean()),
        pop=("Q105", lambda s: (s == 2).mean()))
    x = range(len(g))
    ax.plot(x, g["yall"], "o-", color=PALETTE[1], label="P(y'all)")
    ax.plot(x, g["coke"], "s-", color=PALETTE[2], label="P(coke)")
    ax.plot(x, g["pop"], "^-", color=PALETTE[0], label="P(pop)")
    ax.set_xticks(list(x))
    ax.set_xticklabels([str(int(b.left)) for b in g.index], fontsize=7)
    ax.set_xlabel("longitude band (degrees W)" if key == "long"
                  else "latitude band (degrees N)")
    ax.set_ylabel("probability")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
axes[0].set_title("West -> East")
axes[1].set_title("South -> North")
fig.tight_layout()
fig.savefig(f"{FIG}/continuum_gradient.png", dpi=150)
plt.close(fig)
for key, bins in [("lat", np.arange(24, 51, 4)),
                  ("long", np.arange(-125, -65, 10))]:
    geo["band"] = pd.cut(geo[key], bins=bins)
    print(key)
    print(geo.groupby("band", observed=True).agg(
        yall=("Q050", lambda s: (s == 9).mean()),
        coke=("Q105", lambda s: (s == 3).mean())).round(3).to_string())
print("saved continuum_gradient.png")


lat
           yall   coke
band                  
(24, 28]  0.290  0.221
(28, 32]  0.541  0.433
(32, 36]  0.360  0.322
(36, 40]  0.176  0.152
(40, 44]  0.081  0.054
(44, 48]  0.070  0.044
long
               yall   coke
band                      
(-125, -115]  0.083  0.092
(-115, -105]  0.088  0.108
(-105, -95]   0.343  0.286
(-95, -85]    0.183  0.168
(-85, -75]    0.205  0.129
saved continuum_gradient.png


In [20]:
# The binnings quoted in the report text (10-degree longitude, 4-degree
# latitude bands) -- the figure uses finer bins for shape.
geo2 = df[df["geo_valid"] & (df["long"] > -130) & (df["long"] < -60)
          & (df["lat"] > 24) & (df["lat"] < 50)].copy()
geo2["lat_band"] = pd.cut(geo2["lat"], np.arange(24, 51, 4))
geo2["lon_band"] = pd.cut(geo2["long"], np.arange(-125, -65, 10))
print("by 4-degree latitude bands (south to north)")
print(geo2.groupby("lat_band", observed=True).agg(
    yall=("Q050", lambda s: (s == 9).mean()),
    coke=("Q105", lambda s: (s == 3).mean())).round(3).to_string())
print("\nby 10-degree longitude bands (west to east)")
print(geo2.groupby("lon_band", observed=True).agg(
    yall=("Q050", lambda s: (s == 9).mean()),
    coke=("Q105", lambda s: (s == 3).mean()),
    pop=("Q105", lambda s: (s == 2).mean())).round(3).to_string())


by 4-degree latitude bands (south to north)
           yall   coke
lat_band              
(24, 28]  0.290  0.221
(28, 32]  0.541  0.433
(32, 36]  0.360  0.322
(36, 40]  0.176  0.152
(40, 44]  0.081  0.054
(44, 48]  0.070  0.044

by 10-degree longitude bands (west to east)
               yall   coke    pop
lon_band                         
(-125, -115]  0.083  0.092  0.269
(-115, -105]  0.088  0.108  0.380
(-105, -95]   0.343  0.286  0.313
(-95, -85]    0.183  0.168  0.376
(-85, -75]    0.205  0.129  0.287


The southern cluster is the high end of a gradient, not an island. By
latitude, P(`y'all`) runs 29%, 54%, 36%, 18%, 8%, 7% from the Gulf Coast to the
Canadian border, with P(`coke`) tracking it (43% in the Deep South band against
4% in the north). Longitude adds a wrinkle the map alone would hide: P(`y'all`)
peaks at 34% in the −105..−95° band (Missouri, Arkansas, eastern Texas), and
there is a *secondary* rise of 21% in the −85..−75° band reflecting
Appalachian and Piedmont populations.

So the continuum runs Pacific Coast and Mountain West (`soda`) → Midwest
(`pop`) → Mississippi Valley → Deep South and Appalachia (`coke`, `y'all`),
with sharp local enclaves superimposed rather than replacing it. This is why
the clusters overlap so heavily: three-cluster k-means is a convenient
description of three places where a continuum is locally dense, not a
partition of distinct populations.

### Clusters against Census-style regions

A cluster map is qualitative evidence. The cross-tab below makes the
geographic relationship quantitative, and it is the more convincing of the two:
Cramér's V of 0.523 between cluster and region is far stronger than any
single-question association in the analysis.

In [21]:
# Four Census-style regions, assigned from the STATE code. Dropped
# rows are the 159 junk STATE codes flagged in the audit.
REGIONS = {
    # Census South Atlantic + East South Central: DE, MD and DC included.
    "South": ["AL", "AR", "DE", "DC", "FL", "GA", "KY", "LA", "MD", "MS",
              "NC", "OK", "SC", "TN", "TX", "VA", "WV"],
    "Midwest": ["IL", "IN", "IA", "KS", "MI", "MN", "MO", "NE", "ND", "OH",
                "SD", "WI"],
    "Northeast": ["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"],
    "West": ["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY", "AK", "CA",
             "HI", "OR", "WA"],
}
state = df["STATE"].fillna("")
df["region"] = state.map(lambda s: next(
    (k for k, v in REGIONS.items() if s in v), None))
sub = df.dropna(subset=["region"])
drink = sub[sub["Q105"].isin([1, 2, 3])]
print("Q105 by region (%)")
print((100 * drink.groupby("region")["Q105"].value_counts(normalize=True)
       .unstack().rename(columns={1: "soda", 2: "pop", 3: "coke"}))
      .round(1).to_string())
ct = pd.crosstab(sub["region"], sub["km3"])
print("\ncluster x region row percentages")
print((100 * ct.div(ct.sum(axis=1), axis=0)).round(1).to_string())
chi2_r, p_r, _, _ = chi2_contingency(ct)
V_r = np.sqrt(chi2_r / (ct.values.sum() * (min(ct.shape) - 1)))
print(f"cluster x region: chi2={chi2_r:.0f}, Cramers V={V_r:.3f}, n={ct.values.sum()}")


Q105 by region (%)
Q105       soda   pop  coke
region                     
Midwest    35.6  55.2   9.2
Northeast  81.3  13.2   5.5
South      46.8  11.7  41.4
West       55.1  33.8  11.0

cluster x region row percentages
km3           0     1     2
region                     
Midwest    10.9  24.1  65.0
Northeast  73.2  13.8  13.0
South      20.4  62.5  17.0
West       13.3  12.7  73.9
cluster x region: chi2=24816, Cramers V=0.512, n=47312


The clusters track Census-style regions closely — 73.2% of Northeastern
respondents fall in cluster 0, 65.8% of Southern respondents in cluster 1 — but
they are not coterminous with them, which is the interesting part. Cluster 2
takes 65.0% of the Midwest *and* 73.9% of the West, so it is not a regional
cluster at all but a `pop`-leaning one that spans two disconnected regions
with very different settlement histories. The West is itself split (55.1%
`soda`, 33.8% `pop`), consistent with `soda` owning the Atlantic coast and
`pop` the interior and Pacific.

Clustering output is only interpretable if you know which variables
produced it. Ranking all 67 questions by Cramér's V against the cluster labels
answers that directly, and it separates two kinds of question that look
similar before analysis and behave oppositely after it.

In [22]:
# Cramér's V of each question against the k=3 labels: which questions
# actually carry the regional signal.
tmp2 = df.copy()
tmp2["km3"] = np.load("../tmp/kmeans_k3.npy")
rows = []
# Skip questions nobody varied on -- they cannot separate anything.
for q in [c for c in df.columns if c.startswith("Q")]:
    codes = sorted(c for c in df[q].unique() if c != 0)
    if len(codes) < 2:
        continue
    s2 = tmp2[tmp2[q].isin(codes)]
    ct = pd.crosstab(s2["km3"], s2[q])
    chi2, _, _, _ = chi2_contingency(ct)
    rows.append((q, np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))))
# Rank questions by how much they disagree with the cluster labels.
rows.sort(key=lambda r: -r[1])
print("top separators:", [(q, round(v, 3)) for q, v in rows[:10]])
print("weakest:", [(q, round(v, 3)) for q, v in rows[-3:]])


top separators: [('Q105', np.float64(0.498)), ('Q073', np.float64(0.49)), ('Q106', np.float64(0.431)), ('Q050', np.float64(0.418)), ('Q103', np.float64(0.398)), ('Q076', np.float64(0.388)), ('Q080', np.float64(0.365)), ('Q099', np.float64(0.343)), ('Q074', np.float64(0.338)), ('Q110', np.float64(0.309))]
weakest: [('Q115', np.float64(0.08)), ('Q121', np.float64(0.066)), ('Q057', np.float64(0.04))]


The top of the ranking is dominated by broad north–south variables
(Q105, Q73, Q50, and Q106 `tp'ing`), and the bottom by questions that are
locally interesting but geographically shallow. `hoagie` is a Philadelphia
fact, not a Southern one; `bubbler` is a Wisconsin-and-New-England fact.
Clustering on the full 468-column matrix necessarily down-weights both, because
a Philadelphia enclave is invisible next to 47,000 respondents' worth of
`you guys`.

That is a genuine limitation of this approach rather than a defect in the
data, and it is the clearest argument in this analysis for the alternative
framing in Section 7: if specific enclaves are the object of study, they need
their own geographic units, not one national clustering.

## 9. Stability: restarts and subsampling

### Does the projection hold on unseen respondents?

In [23]:
# Does the projection hold up on respondents the PCA never saw?
idx = np.arange(len(X))
tr, te = train_test_split(idx, test_size=0.30, random_state=42,
                          stratify=pd.to_numeric(df["Q050"], errors="coerce"))
# Fit components on the training split only.
pca_tr = PCA(n_components=10, random_state=42).fit(X[tr])
Z_te = pca_tr.transform(X[te])
# Refit on the held-out split alone, then score its own coordinates against
# the train-derived scores. R^2 near 1 means the training components already
# span what the unseen respondents look like.
pca_te = PCA(n_components=10, random_state=42).fit(X[te])
r2 = r2_score(Z_te, pca_te.transform(X[te])[:, :10], multioutput="raw_values")
print("R^2 of held-out points under train PCs vs PCs refit on them:")
print("  " + "  ".join(f"PC{i + 1}={v:.3f}" for i, v in enumerate(r2)))
print(f"  mean R^2 = {r2.mean():.3f}")
# Sign of an eigenvector is arbitrary, so compare directions up to sign.
cos = [abs(np.dot(pca_tr.components_[i], pca_te.components_[i])) for i in range(10)]
print("  |cosine| between matching components, train vs test refit:")
print("  " + "  ".join(f"PC{i + 1}={c:.3f}" for i, c in enumerate(cos)))
    # Does PC1 still separate the South on respondents it was not fitted on?
south_states = ["AL", "AR", "DE", "DC", "FL", "GA", "KY", "LA", "MD", "MS",
                "NC", "OK", "SC", "TN", "TX", "VA", "WV"]
is_south_te = df.iloc[te]["STATE"].isin(south_states).values
print(f"  PC1 mean on held-out data: South {Z_te[is_south_te, 0].mean():+.3f} "
      f"vs non-South {Z_te[~is_south_te, 0].mean():+.3f}")


R^2 of held-out points under train PCs vs PCs refit on them:
  PC1=0.998  PC2=0.997  PC3=0.998  PC4=0.994  PC5=0.970  PC6=0.968  PC7=0.992  PC8=0.973  PC9=0.958  PC10=0.858
  mean R^2 = 0.971
  |cosine| between matching components, train vs test refit:
  PC1=0.997  PC2=0.998  PC3=0.998  PC4=0.996  PC5=0.982  PC6=0.983  PC7=0.993  PC8=0.983  PC9=0.976  PC10=0.924
  PC1 mean on held-out data: South +0.054 vs non-South -0.023


Stability is what separates a regional pattern from an artifact of one
run. Two perturbations matter and a third is a trap.

Restarts test the algorithm's dependence on initialization; subsampling tests
whether the clusters depend on having every particular respondent. Both are
scored against the full-data partition with the Adjusted Rand Index, which
compares partitions up to label permutation.

Bootstrap resampling with replacement is the obvious third check and is
deliberately *not* reported as a point-wise ARI: duplicated row indices make a
partition comparison on resampled data ill-defined. Subsampling is the cleaner
stability read here.

In [24]:
ref = np.load("../tmp/kmeans_k3.npy")
# Same fit, ten seeds: how much does the partition move?
aris = [adjusted_rand_score(
    ref, KMeans(n_clusters=3, n_init=10,
                random_state=100 + s).fit_predict(X)) for s in range(10)]
print(f"restarts: ARI mean={np.mean(aris):.4f} min={np.min(aris):.4f}")
# Refit on 80% subsamples: ARI/NMI against the full-data labels.
rng = np.random.RandomState(7)
sa, sn = [], []
for t in range(10):
    idx = rng.choice(len(X), size=int(0.8 * len(X)), replace=False)
    lab = KMeans(n_clusters=3, n_init=10, random_state=t).fit_predict(X[idx])
    sa.append(adjusted_rand_score(ref[idx], lab))
    sn.append(normalized_mutual_info_score(ref[idx], lab))
print(f"subsample-80%%: ARI {np.mean(sa):.4f}+/-{np.std(sa):.4f}, "
      f"NMI {np.mean(sn):.4f}+/-{np.std(sn):.4f}")


restarts: ARI mean=0.9440 min=0.8028


subsample-80%%: ARI 0.9720+/-0.0187, NMI 0.9512+/-0.0280


The partition is stable under both perturbations — ARI 0.944 across
restarts and 0.972 ± 0.019 across 80% subsamples. The informative failure is
the *worst* seed: one restart in ten lands in a visibly worse local optimum
(ARI 0.803), which is the concrete argument for reporting `n_init >= 10`
rather than a single fit.

The conclusion is not that k-means is reliable here — the low silhouette says
the cluster boundaries are soft — but that the Southern concentration is not
an artifact of particular rows or a lucky initialization. The smooth
monotonic gradient in Section 8 is the substantive reason to trust it.

## 10. Reality check: hide Q105 and re-cluster

A clustering evaluated only against itself proves nothing. This is the
reality check the instructions ask for, and it is a genuine holdout rather than
a restatement: the beverage question is deleted from the matrix entirely, the
model is refit without it, and then we ask where the respondents who actually
said `coke` ended up.

The logic is that if the southern cluster reflects a real regional
phenomenon, it should capture `coke` drinkers at a much higher rate than
chance — without the beverage question having contributed to the cluster's
existence in the first place.

In [25]:
# Hide the beverage question entirely, then see where coke-drinkers land.
no105 = [c for c in Xb.columns if not c.startswith("Q105=")]
print(f"dropping all Q105 columns: {len(Xb.columns)} -> {len(no105)}")
# Refit with the beverage question entirely absent. If the Southern cluster
# is a real region rather than an artifact, coke-drinkers should still land
# in the y'all-heavy cluster without it having contributed.
lab = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(
    Xb[no105].values.astype(np.float64))
df["cl_no105"] = lab
prof = df.groupby("cl_no105")["Q050"].agg(
    yall_rate=lambda s: (s == 9).mean(), n="size")
print(prof.round(3).to_string())
# Identify the new clusters by their y'all rate, then score the holdout.
south_cl = prof["yall_rate"].idxmax()
for d, name in [(3, "coke"), (2, "pop"), (1, "soda")]:
    m = df["Q105"] == d
    print(f"P(yall-cluster|{name}) = {(df.loc[m, 'cl_no105'] == south_cl).mean():.3f}")


dropping all Q105 columns: 468 -> 458


          yall_rate      n
cl_no105                  
0             0.048  20482
1             0.054  12886
2             0.455  14103
P(yall-cluster|coke) = 0.721
P(yall-cluster|pop) = 0.190
P(yall-cluster|soda) = 0.170


It does. The `y'all`-heavy cluster (45.5% `y'all`, against 4.8% and 5.4%
in the other two) captures 72.1% of the 6,550 respondents who said `coke`,
versus 19% of `pop` respondents and 17% of `soda` respondents — a fourfold
enrichment in a cluster that never saw the beverage question.

One caveat, stated because it weakens the check: part of this signal flows
through Q050 itself, since `y'all` and `coke` are themselves correlated. So
this validates the cluster's *geography* more than it provides fully
independent replication of the beverage finding. It remains meaningful
because the clustering never had access to any beverage information.

## Summary

* Cluster and region agree at Cramér's V = 0.523, stronger than any
  single-question association in the analysis.
* The structure is a continuum: P(`y'all`) runs 7% → 54% from Canada to the
  Gulf Coast, so the clusters are density modes on a gradient rather than
  discrete populations.
* Q50 predicts Q105 strongly but asymmetrically — 85.2% cross-validated
  accuracy against a 15.7% base rate.
* Encoding choices change conclusions: letting no-response become zeros
  produces a spurious non-responder cluster at k = 4.
* Broad axes separate but enclaves hide — `hoagie` and `bubbler` are real but
  geographically shallow, and a national clustering cannot recover them.
* The partition is stable (ARI ≈ 0.95) but barely separated (silhouette ≈
  0.03): overlapping regional dialects, not discrete populations.